# Thermal Balance Enforcement Tests for Redfield Relaxation

This notebook validates that Redfield relaxation channels correctly enforce thermal detailed balance. 

Physical Principle: At thermal equilibrium, transition rates must satisfy:

$$J(-\omega) = J(\omega) \times \exp\left(-\frac{\hbar\omega}{k_B T}\right)$$

where:
- $J(\omega)$ is the spectral density at frequency $\omega$
- $\omega > 0$: downward transition (emission)
- $\omega < 0$: upward transition (absorption)

**Three Thermal Balance Modes:**
1. SKIP: No modification (high-temperature limit)
2. SYMMETRIC: Symmetrizes and enforces detailed balance on all transitions
3. COMPLEMENT: Only fills missing upward transitions from downward ones

**Tests:**
1. Boltzmann factor computation
2. SKIP mode verification
3. SYMMETRIC mode verification
4. COMPLEMENT mode verification
5. Population ratio at equilibrium

In [5]:
%load_ext autoreload
%autoreload 2

import sys
import typing as tp
import os
import math
from importlib import reload

import torch
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import mars
from mars import spin_model, spectra_manager, constants, population, concat
from mars.population import RedfieldRelaxationChannel, CouplingChannelManager
from mars.population.contexts import Context

import matplotlib.pyplot as plt

dtype = torch.float64
device = torch.device("cpu")
complex_dtype = torch.complex128 if dtype == torch.float64 else torch.complex64

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import torch
import numpy as np
import math
from typing import Optional, List, Tuple
from mars.population.contexts import Context, SummedContext
from mars.population import transform
from mars import spin_model
import mars

In [7]:
def create_samples():
    """Create spin system samples for thermal balance tests."""
    g_tensor_1 = spin_model.Interaction((2.02, 2.04, 2.06), dtype=dtype)
    zfs_1 = spin_model.DEInteraction((200 * 1e6, 50 * 1e6), dtype=dtype)
    
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_1],
        electron_electron=[(0, 0, zfs_1)],
        dtype=dtype,
    )
    sample_1 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
    )

    g_tensor_2 = spin_model.Interaction((2.02, 2.14, 2.16), dtype=dtype)
    zfs_2 = spin_model.DEInteraction((200 * 1e6, 70 * 1e6), dtype=dtype)
    
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_2],
        electron_electron=[(0, 0, zfs_2)],
        dtype=dtype
    )
    sample_2 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
    )
    
    return sample_1, sample_2


def get_eigen_basis(sample, field: float):
    """Get eigenbasis vectors and energies for a given magnetic field."""
    magnetic_field = torch.tensor(field, dtype=dtype)
    F, _, _, Gz = sample.get_hamiltonian_terms()
    H = F + Gz * magnetic_field
    H = H.unsqueeze(-3)
    energies, vectors = torch.linalg.eigh(H)
    return vectors, energies


def create_operator(sample, operator_dtype=complex_dtype):
    """Create a random Hermitian operator for coupling."""
    N = sample.spin_system_dim
    operator = torch.randn((N, N), dtype=operator_dtype)
    operator = (operator + operator.transpose(-2, -1).conj()) / 2
    return operator

In [8]:
"""
Thermal Equilibrium Tests for RedfieldRelaxationChannel
========================================================

Self-contained tests — no dependency on the `mars` package internals.
Copy the RedfieldRelaxationChannel class (and its helpers) into scope,
then run:  python test_thermal_equilibrium_redfield.py

Physical ground truth
---------------------
Detailed balance (Kubo-Martin-Schwinger condition):
    J(-ω) = J(ω) × exp(-ℏω / k_B T)

where  ω > 0  is a downward (emission) transition frequency in rad/s,
and    ω < 0  is an upward  (absorption) transition.

In the matrix convention used here:
    omega_hz[i, j] = E_j - E_i          [Hz]
    J[i, j]  = spectral density for transition j→i
    J[i, j] / J[j, i]  =  exp(-h/k_B × omega_hz[i,j] / T)
                        =  exp(-energy_diff_K[i,j] / T)

At equilibrium the population vector P_eq satisfies:
    L @ P_eq = 0
where L is the full Liouvillian (rate matrix with diagonal drain):
    L[a, a]  = -sum_{c≠a} W[c, a]
    L[a, c]  =  W[a, c]              for a ≠ c
and P_eq[i] = exp(-E_i/k_B T) / Z.
"""

import math
import torch

# ---------------------------------------------------------------------------
# Minimal physical constants (replace with your constants module if preferred)
# ---------------------------------------------------------------------------
H_OVER_KB = 4.799243073e-11   # h / k_B  [K / Hz]  (= ℏ / k_B × 2π)

def hz_to_K(omega_hz: torch.Tensor) -> torch.Tensor:
    """Convert frequency in Hz to temperature units: ΔE/k_B [K]."""
    return omega_hz * H_OVER_KB


# ---------------------------------------------------------------------------
# Helpers to build toy systems without any spin-model dependency
# ---------------------------------------------------------------------------

def make_two_level(delta_hz: float, dtype=torch.float64):
    """
    Return (energies, operator) for a simple two-level system.

    energies : shape (N,)   [Hz]
    operator  : shape (N,N) Hermitian coupling operator
    """
    energies = torch.tensor([0.0, delta_hz], dtype=dtype)
    # Random Hermitian operator (fixed seed for reproducibility)
    torch.manual_seed(42)
    op_raw = torch.randn(2, 2, dtype=dtype)
    op = (op_raw + op_raw.T) / 2
    return energies, op


def make_three_level(dtype=torch.float64):
    """Three-level ladder with non-degenerate gaps."""
    energies = torch.tensor([0.0, 5e9, 8.5e9], dtype=dtype)   # Hz
    torch.manual_seed(7)
    op_raw = torch.randn(3, 3, dtype=dtype)
    op = (op_raw + op_raw.T) / 2
    return energies, op


# ---------------------------------------------------------------------------
# Spectral density functions for testing
# ---------------------------------------------------------------------------

def flat_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
    """J(ω) = 1  (constant, symmetric, no thermal bias)."""
    return torch.ones_like(omega_rad_s)


def asymmetric_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
    """J(ω) = 1 for ω > 0, 0.5 for ω ≤ 0  (deliberately breaks symmetry)."""
    return torch.where(omega_rad_s > 0,
                       torch.ones_like(omega_rad_s),
                       0.5 * torch.ones_like(omega_rad_s))


def downward_only_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
    """J(ω) = 1 for ω > 0, 0 for ω ≤ 0  (only downward/emission rates)."""
    return torch.where(omega_rad_s > 0,
                       torch.ones_like(omega_rad_s),
                       torch.zeros_like(omega_rad_s))


# ---------------------------------------------------------------------------
# Minimal stub of RedfieldRelaxationChannel
# (paste the real class here if running standalone, or import it)
# ---------------------------------------------------------------------------
try:
    from mars.population import RedfieldRelaxationChannel  # type: ignore
except ImportError:
    raise ImportError(
        "Could not import RedfieldRelaxationChannel from mars.population. "
        "Either install the mars package or paste the class definition above "
        "this import block."
    )


# ---------------------------------------------------------------------------
# Helper: build channel and get spectral density matrix
# ---------------------------------------------------------------------------

def _build_channel(op: torch.Tensor, spec_func, mode: str) -> RedfieldRelaxationChannel:
    ch = RedfieldRelaxationChannel(
        operator_components=[(op, None)],
        spectral_density_func=spec_func,
        thermal_balance_mode=mode,
    )
    ch.post_init(eigen_basis_flag=True, secular=True)
    return ch


def _spec_matrix(ch, energies, temperature):
    return ch._get_spectral_density_matrix(energies, temperature)


def _omega_hz(energies):
    """omega_hz[i,j] = E_j - E_i  [Hz]."""
    return energies.unsqueeze(-2) - energies.unsqueeze(-1)


def _boltzmann_matrix(omega_hz, temperature):
    """exp(-h/k_B × omega_hz / T) = exp(-energy_diff_K / T)."""
    return torch.exp(-hz_to_K(omega_hz) / temperature)


# ===========================================================================
# TEST 1: Boltzmann factor sanity checks
# ===========================================================================

def test_boltzmann_factors():
    """
    Basic sanity checks on the exp(-hω/kT) factor:
      - diagonal entries must equal 1 (zero energy gap)
      - B[i,j] × B[j,i] = 1  (reciprocity)
      - B[i,j] ≤ 1 when omega_hz[i,j] > 0  (downward: Boltzmann suppresses absorption)
      - B[i,j] ≥ 1 when omega_hz[i,j] < 0  (upward:   Boltzmann enhances emission)
    """
    energies, _ = make_three_level()
    atol = 1e-10

    for T in [4.2, 77.0, 300.0]:
        temperature = torch.tensor(T, dtype=torch.float64)
        omega = _omega_hz(energies)
        B = _boltzmann_matrix(omega, temperature)

        # Diagonal = 1
        diag = torch.diagonal(B)
        assert torch.allclose(diag, torch.ones_like(diag), atol=atol), \
            f"T={T}: diagonal B[i,i] ≠ 1"

        # Reciprocity: B[i,j] * B[j,i] = 1
        product = B * B.T
        assert torch.allclose(product, torch.ones_like(product), atol=atol), \
            f"T={T}: B[i,j] * B[j,i] ≠ 1"

        # Downward transitions (omega > 0): emission, B ≤ 1
        down_mask = omega > 0
        assert torch.all(B[down_mask] <= 1.0 + atol), \
            f"T={T}: B should be ≤ 1 for downward transitions"

        # Upward transitions (omega < 0): absorption, B ≥ 1
        up_mask = omega < 0
        assert torch.all(B[up_mask] >= 1.0 - atol), \
            f"T={T}: B should be ≥ 1 for upward transitions"

    print("PASS  test_boltzmann_factors")


# ===========================================================================
# TEST 2: SKIP mode — spectral density is unmodified
# ===========================================================================

def test_skip_mode_unmodified():
    """
    SKIP mode must return exactly the raw spectral density with no modification.
    We use a flat J so the expected output is a matrix of ones.
    """
    energies, op = make_three_level()
    temperature = torch.tensor(300.0, dtype=torch.float64)
    ch = _build_channel(op, flat_spectral_density, "skip")
    J = _spec_matrix(ch, energies, temperature)
    expected = torch.ones_like(J)
    assert torch.allclose(J, expected, atol=1e-8), \
        "SKIP mode should return J unchanged"
    print("PASS  test_skip_mode_unmodified")


def test_skip_mode_no_thermal_bias():
    """
    With constant J(ω)=1 and SKIP, the rate matrix W must be symmetric
    (no thermal preference for either direction).
    """
    energies, op = make_three_level()
    temperature = torch.tensor(300.0, dtype=torch.float64)
    ch = _build_channel(op, flat_spectral_density, "skip")
    W = ch.transition_probabilities(None, None, energies.unsqueeze(-2), temperature)
    assert torch.allclose(W, W.transpose(-2, -1), atol=1e-8), \
        "SKIP + constant J should give symmetric rate matrix"
    print("PASS  test_skip_mode_no_thermal_bias")


# ===========================================================================
# TEST 3: SYMMETRIC mode — detailed balance in spectral density matrix
# ===========================================================================

def test_symmetric_mode_detailed_balance_spectral():
    """
    After SYMMETRIC processing, the spectral density matrix must satisfy
    the KMS condition at every off-diagonal entry:

        J[j,i] / J[i, j] =  exp(-h × omega_hz[i,j] / k_B T)
                          =  exp(-energy_diff_K[i,j] / T)

    We use an asymmetric raw J so that the mode is actually doing work.
    """
    energies, op = make_three_level()
    atol = 1e-6

    for T in [4.2, 77.0, 300.0]:
        temperature = torch.tensor(T, dtype=torch.float64)
        ch = _build_channel(op, asymmetric_spectral_density, "symmetric")
        J = _spec_matrix(ch, energies, temperature)

        omega = _omega_hz(energies)
        B_expected = _boltzmann_matrix(omega, temperature).transpose(-2, -1)   # exp(-hω/kT)

        N = energies.shape[-1]
        for i in range(N):
            for j in range(N):
                if i == j:
                    continue
                j_ij = J[i, j].item()
                j_ji = J[j, i].item()
                if j_ji < 1e-30:
                    continue
                ratio = j_ij / j_ji
                expected = B_expected[i, j].item()
                assert abs(ratio - expected) < atol + 0.01 * abs(expected), \
                    (f"T={T}: SYMMETRIC detailed balance violated at [{i},{j}]. "
                     f"J[i,j]/J[j,i]={ratio:.6f}, exp(-hω/kT)={expected:.6f}")

    print("PASS  test_symmetric_mode_detailed_balance_spectral")


def test_symmetric_mode_detailed_balance_rates():
    """
    Transition rates W must also satisfy detailed balance:

        W[a,c] / W[c,a]  =  exp(-h × (E_c - E_a) / k_B T)

    for all pairs (a, c) with a ≠ c, where 'a' is the final state.
    (i.e. W[a,c] is the rate c→a.)
    """
    energies, op = make_three_level()
    temperature = torch.tensor(77.0, dtype=torch.float64)
    ch = _build_channel(op, asymmetric_spectral_density, "symmetric")
    W = ch.transition_probabilities(None, None, energies.unsqueeze(-2), temperature)

    omega = _omega_hz(energies)
    B_expected = _boltzmann_matrix(omega, temperature).transpose(-2, -1)

    N = energies.shape[-1]
    atol_rel = 5e-5
    checked = 0
    for a in range(N):
        for c in range(N):
            if a == c:
                continue
            w_ac = W[0, a, c].item()
            w_ca = W[0, c, a].item()
            if w_ac < 1e-30 or w_ca < 1e-30:
                continue
            ratio = w_ac / w_ca
            # W[a,c] = rate c→a; omega_hz[a,c] = E_c - E_a
            expected = B_expected[a, c].item()
            assert abs(ratio - expected) < atol_rel + 0.01 * abs(expected), \
                (f"SYMMETRIC rate detailed balance violated W[{a},{c}]/W[{c},{a}] "
                 f"= {ratio:.6f}, expected {expected:.6f}")
            checked += 1

    assert checked > 0, "No valid transition pairs found — check the test setup"
    print(f"PASS  test_symmetric_mode_detailed_balance_rates ({checked} pairs checked)")


# ===========================================================================
# TEST 4: COMPLEMENT mode — fills only zero upward entries
# ===========================================================================

def test_complement_mode_fills_upward_zeros():
    """
    Starting from a spectral density that is zero for all ω ≤ 0
    (downward_only_spectral_density), the COMPLEMENT mode must fill
    every zero upward entry using:

        J[i,j]  =  J[j,i] × exp(-energy_diff_K[j,i] / T)
                 =  J[j,i] × exp(+energy_diff_K[i,j] / T)

    for entries where omega_hz[i,j] < 0  (upward transition).
    Downward entries (omega_hz[i,j] > 0) must remain unchanged.
    """
    energies, op = make_three_level()
    temperature = torch.tensor(77.0, dtype=torch.float64)
    ch = _build_channel(op, downward_only_spectral_density, "complement")
    J = _spec_matrix(ch, energies, temperature)

    omega = _omega_hz(energies)
    N = energies.shape[-1]
    atol = 1e-6

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            omg = omega[i, j].item()
            j_ij = J[i, j].item()
            j_ji = J[j, i].item()
            
            if omg > 0:
                # Downward: preserved, must be > 0 (raw J = 1 for ω > 0)
                assert j_ij > 0, \
                    f"Downward J[{i},{j}] should be preserved (> 0)"
            elif omg < 0:
                # Upward: must be filled and satisfy detailed balance
                assert j_ij > 0, \
                    f"Upward J[{i},{j}] should be filled by COMPLEMENT (was 0)"
                # J[i,j] = J[j,i] * exp(-energy_diff_K[j,i] / T)
                # omega_hz[j,i] = E_i - E_j > 0 (downward partner)
                energy_diff_K_ji = hz_to_K(omega[j, i]).item()
                expected = j_ji * math.exp(-energy_diff_K_ji / temperature.item())
                assert abs(j_ij - expected) < atol + 0.01 * expected, \
                    (f"COMPLEMENT J[{i},{j}] = {j_ij:.6e}, "
                     f"expected {expected:.6e} from detailed balance")

    print("PASS  test_complement_mode_fills_upward_zeros")


def test_complement_mode_preserves_nonzero_upward():
    """
  - Upward transitions (omega < 0) are preserved when nonzero.
  - Downward transitions (omega > 0) are overwritten to satisfy
    detailed balance relative to the preserved upward partner.
  - Using flat_spectral_density (all ones) ensures both directions
    are initially nonzero, triggering the preservation/modification logic.
    """
    energies, op = make_three_level()
    temperature = torch.tensor(77.0, dtype=dtype)
    ch = _build_channel(op, flat_spectral_density, "complement")
    J = _spec_matrix(ch, energies, temperature)

    omega = _omega_hz(energies)
    N = energies.shape[-1]
    atol = 1e-8

    for i in range(N):
        for j in range(N):
            if i == j:
                # Diagonal entries should remain unchanged (1.0)
                assert torch.isclose(J[i, j], torch.tensor(1.0, dtype=dtype), atol=atol), \
                    f"Diagonal J[{i},{i}] should remain 1.0"
                continue

            omg = omega[i, j].item()
            j_ij = J[i, j].item()
            j_ji = J[j, i].item()

            if omg < 0:
                assert torch.isclose(J[i, j], torch.tensor(1.0, dtype=dtype), atol=atol), \
                    f"Upward J[{i},{j}] should be preserved as 1.0, got {j_ij}"
            else:
                energy_diff_K = hz_to_K(torch.tensor(omg, dtype=dtype)).item()
                expected = j_ji * math.exp(-energy_diff_K / temperature.item())

    print("PASS  test_complement_mode_preserves_nonzero_upward")


# ===========================================================================
# TEST 5: Thermal equilibrium of population dynamics
# ===========================================================================

def _boltzmann_distribution(energies: torch.Tensor, temperature: torch.Tensor) -> torch.Tensor:
    """Normalised Boltzmann distribution P_eq[i] = exp(-E_i h/k_B T) / Z."""
    exponents = -hz_to_K(energies) / temperature
    exponents = exponents - exponents.max()   # numerical stability
    p = torch.exp(exponents)
    return p / p.sum()


def _build_liouvillian(W: torch.Tensor) -> torch.Tensor:
    """
    Build the full rate matrix (Liouvillian) L from the off-diagonal rate
    matrix W (where W[a,c] = rate c→a, diagonal is 0):

        L[a, c] = W[a, c]                for a ≠ c
        L[c, c] = -sum_{a ≠ c} W[a, c]  (probability drain out of c)
    """
    drain = W.sum(dim=-2)          # total rate out of each state
    L = W.clone()
    idx = torch.arange(W.shape[-1])
    L[..., idx, idx] = -drain
    return L


def test_symmetric_equilibrium_steady_state():
    """
    With SYMMETRIC thermal balance, the Boltzmann vector P_eq must be in
    the null-space of the full Liouvillian L:

        L @ P_eq ≈ 0

    This is the definitive check that the dynamics drive the system to
    the correct thermal equilibrium.
    """
    energies, op = make_three_level()
    temperature = torch.tensor(77.0, dtype=torch.float64)
    ch = _build_channel(op, asymmetric_spectral_density, "symmetric")
    W = ch.transition_probabilities(None, None, energies.unsqueeze(-2), temperature)
    print(W)
    L = _build_liouvillian(W)

    P_eq = _boltzmann_distribution(energies, temperature)
    dP_dt = L @ P_eq
    print(P_eq)

    atol = 1e-5
    assert torch.allclose(dP_dt, torch.zeros_like(dP_dt), atol=atol), \
        (f"P_eq is not steady state under SYMMETRIC Liouvillian. "
         f"Max |dP/dt| = {dP_dt.abs().max().item():.2e}")
    print("PASS  test_symmetric_equilibrium_steady_state")


def test_complement_equilibrium_steady_state():
    """Same steady-state test for COMPLEMENT mode."""
    energies, op = make_three_level()
    temperature = torch.tensor(77.0, dtype=torch.float64)
    ch = _build_channel(op, downward_only_spectral_density, "complement")
    W = ch.transition_probabilities(None, None, energies.unsqueeze(-2), temperature)
    L = _build_liouvillian(W)

    P_eq = _boltzmann_distribution(energies, temperature)
    dP_dt = L @ P_eq

    atol = 1e-5
    assert torch.allclose(dP_dt, torch.zeros_like(dP_dt), atol=atol), \
        (f"P_eq is not steady state under COMPLEMENT Liouvillian. "
         f"Max |dP/dt| = {dP_dt.abs().max().item():.2e}")
    print("PASS  test_complement_equilibrium_steady_state")


def test_skip_does_not_achieve_equilibrium_unless_flat():
    """
    SKIP with an asymmetric J should NOT satisfy detailed balance,
    so the Boltzmann distribution is NOT a steady state.
    (Confirms the other modes are actually doing something.)
    """
    energies, op = make_three_level()
    temperature = torch.tensor(77.0, dtype=torch.float64)
    ch = _build_channel(op, asymmetric_spectral_density, "skip")
    W = ch.transition_probabilities(None, None, energies.unsqueeze(-2), temperature)
    print(W)
    L = _build_liouvillian(W)

    P_eq = _boltzmann_distribution(energies, temperature)
    dP_dt = L @ P_eq

    # With asymmetric J and SKIP, the Boltzmann vector should NOT be steady
    is_steady = torch.allclose(dP_dt, torch.zeros_like(dP_dt), atol=1e-5)
    assert not is_steady, \
        ("SKIP + asymmetric J should NOT yield Boltzmann steady state. "
         "Either the test J is too symmetric, or thermal balance is being "
         "applied despite SKIP mode.")
    print("PASS  test_skip_does_not_achieve_equilibrium_unless_flat")


# ===========================================================================
# TEST 6: omega conversion — catches the operator-precedence bug
# ===========================================================================

def test_omega_conversion_sign_and_scale():
    """
    Verify that the spectral density function receives frequencies in rad/s,
    not some scrambled multiple.  We use a spectral density that records its
    input and check that the magnitude is 2π × omega_hz.
    """
    received = []

    def recording_spec(omega_rad_s, temperature: tp.Optional[torch.Tensor]):
        received.append(omega_rad_s.clone())
        return torch.ones_like(omega_rad_s)

    energies, op = make_two_level(delta_hz=1e10)
    temperature = torch.tensor(300.0, dtype=torch.float64)
    ch = _build_channel(op, recording_spec, "skip")
    _ = _spec_matrix(ch, energies, temperature)

    assert len(received) > 0, "spectral_density_func was never called"
    omega_passed = received[0]

    # The off-diagonal frequency for a two-level system with gap delta_hz
    delta_hz = 1e10
    expected_rad_s = 2 * math.pi * delta_hz
    # Check the off-diagonal element (0,1): omega_hz[0,1] = E_1 - E_0 = delta_hz
    # so omega_rad_s[0,1] should be 2π × delta_hz
    off_diag_freq = omega_passed[0, 1].abs().item()
    rel_err = abs(off_diag_freq - expected_rad_s) / expected_rad_s
    assert rel_err < 1e-6, \
        (f"Spectral density received ω = {off_diag_freq:.4e} rad/s, "
         f"expected 2π × {delta_hz:.4e} = {expected_rad_s:.4e} rad/s. "
         f"Relative error: {rel_err:.2e}. "
         "Check operator precedence: use  omega_hz * 2 * math.pi,  "
         "not  omega_hz / 2 * math.pi.")
    print("PASS  test_omega_conversion_sign_and_scale")


# ===========================================================================
# TEST 7: mode initialisation — invalid strings and types
# ===========================================================================

def test_invalid_mode_raises():
    """Invalid mode strings and wrong types must raise informative errors."""
    energies, op = make_two_level(delta_hz=1e9)

    try:
        RedfieldRelaxationChannel(
            operator_components=[(op, None)],
            spectral_density_func=flat_spectral_density,
            thermal_balance_mode="nonsense",
        )
        assert False, "Should have raised ValueError for unknown mode string"
    except ValueError:
        pass

    try:
        RedfieldRelaxationChannel(
            operator_components=[(op, None)],
            spectral_density_func=flat_spectral_density,
            thermal_balance_mode=42,
        )
        assert False, "Should have raised TypeError for integer mode"
    except TypeError:
        pass

    print("PASS  test_invalid_mode_raises")


# ===========================================================================
# Run all tests
# ===========================================================================

def run_all():
    print("=" * 60)
    print("Redfield Thermal Equilibrium Test Suite")
    print("=" * 60)
    test_boltzmann_factors()
    test_skip_mode_unmodified()
    test_skip_mode_no_thermal_bias()
    test_symmetric_mode_detailed_balance_spectral()
    test_symmetric_mode_detailed_balance_rates()
    test_complement_mode_fills_upward_zeros()
    test_complement_mode_preserves_nonzero_upward()
    test_symmetric_equilibrium_steady_state()
    test_complement_equilibrium_steady_state()
    test_skip_does_not_achieve_equilibrium_unless_flat()
    test_omega_conversion_sign_and_scale()
    test_invalid_mode_raises()
    print("=" * 60)
    print("ALL TESTS PASSED")
    print("=" * 60)


if __name__ == "__main__":
    run_all()

Redfield Thermal Equilibrium Test Suite
PASS  test_boltzmann_factors
PASS  test_skip_mode_unmodified
PASS  test_skip_mode_no_thermal_bias
PASS  test_symmetric_mode_detailed_balance_spectral
PASS  test_symmetric_mode_detailed_balance_rates (6 pairs checked)
PASS  test_complement_mode_fills_upward_zeros
PASS  test_complement_mode_preserves_nonzero_upward
tensor([[[0.0000, 0.0202, 0.0657],
         [0.0202, 0.0000, 0.0214],
         [0.0654, 0.0213, 0.0000]]], dtype=torch.float64)
tensor([0.3343, 0.3332, 0.3325], dtype=torch.float64)
PASS  test_symmetric_equilibrium_steady_state
PASS  test_complement_equilibrium_steady_state
tensor([[[0.0000, 0.0269, 0.0874],
         [0.0135, 0.0000, 0.0285],
         [0.0437, 0.0142, 0.0000]]], dtype=torch.float64)
PASS  test_skip_does_not_achieve_equilibrium_unless_flat
PASS  test_omega_conversion_sign_and_scale
PASS  test_invalid_mode_raises
ALL TESTS PASSED
